# מפגש 27 — SVM והשוואת מודלים | מעבדת תלמיד
הדגמת המורה היא עם ארבעה מאפיינים, ואילו אתם עובדים על שניים: `bill_depth_mm` ו־`body_mass_g` לניבוי `sex`. את ערך C=1, kernel='linear' ואת הפיצול random_state=17 קובעים מראש. המטרה היא להבין שוליים, מטריצת בלבול וציון החלטה — **בלי** לבחור לפי Test.

## 0. חישוב ידני וציון חתום
ניקח ציונים דמיוניים [−0.2, 0.7]. הם אינם הסתברויות. איזה מהם נמצא בצד החיובי? מדוע 0.7 לא אומר 70%?

In [ ]:
import numpy as np
synthetic_scores=np.array([-.2,.7])
sides=np.where(synthetic_scores>=???,1,0)
assert sides.tolist()==[0,1]
print(sides)

## 1. טעינה, פיצול ובדיקת יעד
מוציאים שורות עם `sex` חסר, כי אין להשלים תוויות מבלי לדעת אותן. התוויות הן female/male. שני מאפיינים נשארים עם ערכים חסרים, ואותם יש להשלים **בתוך** Pipeline.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report,confusion_matrix
url="https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df=pd.read_csv(url)
model_df=df.dropna(subset=[???]).copy()
features=[???,???]
X=model_df[features].copy()
y=model_df[???].copy()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=???,random_state=???,stratify=y)
assert list(X.columns)==["bill_depth_mm","body_mass_g"]
assert set(X_train.index).isdisjoint(set(X_test.index))
assert sorted(y.unique().tolist())==["female","male"]

## 2. אימון SVM ללא זליגה
שימו לב לסדר: השלמת ערכים חסרים → תקנון → SVC. את ה־fit מבצעים על Train בלבד. `C=1` נקבע מראש ולא בעקבות הצגת תוצאות Test.

In [ ]:
model=make_pipeline(SimpleImputer(strategy=???),StandardScaler(),SVC(kernel=???,C=???))
model.???(X_train,y_train)
predictions=model.???(X_test)
assert model.named_steps["svc"].kernel=="linear"
assert model.named_steps["svc"].C==1.
assert len(predictions)==len(y_test)

## 3. מטריצת בלבול ופרשנות
השתמשו ב־`labels=['female','male']`, אמת בשורות וחיזוי בעמודות. ספרו טעויות לפי תאים מחוץ לאלכסון; אין להסיק סיבתיות.

In [ ]:
labels=["female","male"]
matrix=confusion_matrix(???,???,labels=???)
assert matrix.shape==(2,2)
assert matrix.sum()==len(y_test)
report=classification_report(y_test,predictions,labels=labels,zero_division=0)
print(pd.DataFrame(matrix,index=labels,columns=labels))
print(report)

## 4. decision_function לעומת הסתברות
החזרת ציון חתום במודל בינארי; מחלקת הציון החיובי נקבעת מתוך `classes_[1]`. אין לתרגם את הציון ל־'אחוזי ביטחון'.

In [ ]:
scores=model.???(X_test)
classes=model.named_steps["svc"].???
assert list(classes)==["female","male"]
assert len(scores)==len(predictions)
assert np.isfinite(scores).all()
sign_predictions=np.where(scores>=???,classes[1],classes[0])
assert np.array_equal(sign_predictions,predictions)
print("First signed scores:",scores[:5],"Positive-score class:",classes[1])

## 5. חשיבה ביקורתית וכרטיס הגשה
מהם שוליים (Margin)? מדוע C גבוה או נמוך אינו מבטיח דיוק? כיצד אפשר לבחור C על ידי אימות צולב על Train בלבד? האם `decision_function=0.8` היא הסתברות? מה מגביל השוואה מול דוגמת המורה שבה המאפיינים והפיצול שונים? כתבו נימוק והצעת ניסוי הוגן, ללא שינוי C על סמך Test.